# Aralin 05 - Agentic RAG


## Setup

Ipinapakita ng notebook na ito ang Agentic RAG (Retrieval-Augmented Generation) pattern gamit ang Microsoft Agent Framework.

**Mga Kinakailangan:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — ang iyong Azure AI Search service endpoint
- `AZURE_SEARCH_API_KEY` — ang iyong Azure AI Search API key
- Azure OpenAI deployment na nakonpigura sa pamamagitan ng mga environment variables
- Azure CLI na naka-authenticate (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Ano ang Agentic RAG?

Ang Tradisyunal na RAG ay sumusunod sa isang nakapirming proseso: kunin ang mga dokumento, pagkatapos ay bumuo ng sagot. Ang **Agentic RAG** ay mas sumusulong sa pamamagitan ng pagbibigay sa ahente ng awtonomiya upang magpasya **kailan** at **paano** kukuha ng impormasyon.

Sa Agentic RAG, ang ahente ay maaaring:
- **Magpasya** kung kailangan ba muna ng pagkukuha bago sumagot sa isang tanong
- **Pumili** kung aling pinagkukunan ng datos o kasangkapan ang tatanungin
- **Suriin** ang mga nakuha na resulta at magsagawa ng karagdagang pagkukuha kung hindi sapat ang unang pagtatangka
- **Pagsamahin** ang impormasyon mula sa maraming hakbang ng pagkukuha sa isang magkakaugnay na sagot

Ito ay nagpapalakas ng kakayahan at katumpakan ng ahente kumpara sa isang static na proseso ng retrieve-then-generate.


## Paglikha ng Isang Kasangkapan sa Paghahanap

Sa Agentic RAG, ang mga panlabas na pinagkukunan ng datos ay binalot bilang **tools** na maaaring tawagin ng ahente kapag kinakailangan. Pinapayagan nito ang ahente na ituring ang pagkuha bilang isa lamang pang aksyon na maaari nitong gawin, sa halip na isang obligadong hakbang.

Sa ibaba ay tinutukoy namin ang isang base ng kaalaman tungkol sa paglalakbay at inilalantad ito bilang isang kasangkapan na maaaring tawagin ng ahente upang maghanap ng impormasyon tungkol sa destinasyon.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Pagbuo ng RAG Agent

Ngayon ay gagawa tayo ng isang agent na inatasang **laging kumuha ng impormasyon bago sumagot**. Ginagamit ng agent ang tool na `search_travel_knowledge` upang patatagin ang mga tugon nito mula sa knowledge base sa halip na umasa sa sariling training data nito.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iterative Retrieval — Ang Maker-Checker na Pattern

Isang pangunahing bentahe ng Agentic RAG ay ang **paulit-ulit na retrieval**. Maaaring magsagawa ang ahente ng maraming rounds ng paghahanap upang beripikahin, pinuhin, o palawakin ang mga unang natuklasan — katulad ng isang "maker-checker" na workflow:

1. **Maker na hakbang**: Kinukuha ng ahente ang paunang impormasyon at gumagawa ng draft ng sagot.
2. **Checker na hakbang**: Nagsasagawa ang ahente ng karagdagang retrieval upang beripikahin ang mga detalye o punan ang mga puwang.

Sa ibaba, tinatanong ang ahente ng isang tanong na nangangailangan ng paghahambing ng maraming destinasyon, kaya hinihikayat itong maghanap nang ilang beses.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Summary

Sa araling ito natutunan mo kung paano bumuo ng isang **Agentic RAG** system gamit ang Microsoft Agent Framework:

- Pinapayagan ng **Agentic RAG** ang mga agent na awtonomong magpasya kung kailan kukuha ng impormasyon, kaya ang retrieval ay nagiging dynamic sa halip na nakatakda.
- **Mga tools bilang mga pinagmumulan ng datos**: Ang mga panlabas na knowledge bases (tulad ng Azure AI Search) ay ini-wrap bilang mga tools na maaaring tawagin ng agent.
- **Iteratibong retrieval**: Pinapagana ng maker-checker pattern ang agent na magsagawa ng maraming retrieval rounds — paghahanap, pag-verify, at pagpapino — bago magbigay ng pangwakas na sagot.

Sa produksyon, papalitan mo ang in-memory na `TRAVEL_KNOWLEDGE_BASE` ng totoong Azure AI Search index upang hawakan ang malakihang retrieval ng travel documents.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
